# Task 3: Temporal Negation

**Task:** `temporal_negation`  |  **Metric:** Accuracy

Dataset: `/kaggle/input/datasets/zacharymaronek/adversarial-temporal-crusher-dataset/temporal_negation.csv`

A fact has changed/ended. Naive latest-mention retrieval fails.

In [ ]:
# @title Cell 1: Install kbench
!pip install kaggle-benchmarks -q
print("done")

In [ ]:
# @title Cell 2: Load Dataset
import pandas as pd, os
os.environ["RENDER_SUBRUNS"] = "False"

df = pd.read_csv("/kaggle/input/datasets/zacharymaronek/adversarial-temporal-crusher-dataset/temporal_negation.csv")
df = df[["question_id", "prompt", "correct_answer", "naive_answer"]]
for col in df.columns:
    df[col] = df[col].apply(lambda x: str(x) if pd.notna(x) else "")
print(f"{len(df)} questions")
print(df.head(2).to_string())

In [ ]:
# @title Cell 3: Define Task
import kaggle_benchmarks as kbench

@kbench.task(name="temporal_negation", store_task=False)
def temporal_negation(llm, question_id: str, prompt: str, correct_answer: str, naive_answer: str) -> bool:
    response = llm.prompt(f"{prompt}\n\nAnswer with only the value (e.g. v17). Do not explain.")
    return correct_answer.lower() in response.lower()

In [ ]:
# @title Cell 4: Evaluate
results = temporal_negation.evaluate(
    llm=[kbench.llm],
    evaluation_data=df,
    stop_condition=lambda runs: len(runs) == len(df),
    max_attempts=10,
    retry_delay=30,
    n_jobs=4,
    timeout=120,
)
rdf = results.as_dataframe()
print(f"Accuracy: {rdf['result'].mean()*100:.1f}%")
display(rdf.head(10))

In [ ]:
# @title Cell 5: Save Submission
rdf.to_csv("/content/submission_temporal_negation.csv", index=False)
print(f"Saved {len(rdf)} rows")